# Redrob preprocess — Colab GPU + Gemini

- **CUDA GPU** → fast BGE embeddings (~15–45 min for 100K)
- **Gemini API** → JD parse + archetype labels
- Outputs → **`artifacts/`** (download zip and copy to your local repo)

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Gemini auth:** Upload your ADC credentials JSON (`authorized_user` with `refresh_token`) or set Colab Secret `GOOGLE_ADC_JSON`

In [ ]:
# Optional: mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/ashokbugude/recruiter-candidate.git'
if not Path('/content/recruiter-candidate/rank.py').exists():
    !git clone {REPO_URL} /content/recruiter-candidate

PROJECT_ROOT = Path('/content/recruiter-candidate')
assert (PROJECT_ROOT / 'rank.py').exists(), f'Project not found: {PROJECT_ROOT}'
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT.resolve())

In [ ]:
!pip install -q -r colab/requirements.txt

In [ ]:
from pathlib import Path
candidates_path = PROJECT_ROOT / 'challenge' / 'candidates.jsonl'
if not candidates_path.exists():
    from google.colab import files
    print('Upload candidates.jsonl (~465 MB)')
    uploaded = files.upload()
    (PROJECT_ROOT / 'challenge').mkdir(parents=True, exist_ok=True)
    for name, data in uploaded.items():
        candidates_path.write_bytes(data)
    print('Saved', candidates_path)
else:
    print('Found', candidates_path, f'({candidates_path.stat().st_size // 1_048_576} MB)')

In [ ]:
USE_GEMINI = True
FORCE = True

import json
import os
from pathlib import Path

ADC_PATH = PROJECT_ROOT / 'credentials' / 'adc.json'

if USE_GEMINI:
    ADC_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not ADC_PATH.exists():
        try:
            from google.colab import userdata
            adc = json.loads(userdata.get('GOOGLE_ADC_JSON'))
            ADC_PATH.write_text(json.dumps(adc), encoding='utf-8')
            print('Wrote ADC credentials from Colab Secret')
        except Exception:
            from google.colab import files
            print('Upload your ADC credentials JSON (authorized_user with refresh_token)')
            uploaded = files.upload()
            for name, data in uploaded.items():
                ADC_PATH.write_bytes(data)
            print('Saved', ADC_PATH)
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(ADC_PATH)
    os.environ.setdefault('GOOGLE_CLOUD_LOCATION', 'us-central1')
    adc_data = json.loads(ADC_PATH.read_text(encoding='utf-8'))
    if adc_data.get('quota_project_id'):
        os.environ.setdefault('GOOGLE_CLOUD_PROJECT', adc_data['quota_project_id'])
    os.environ['REDROB_SKIP_GEMINI'] = '0'
    os.environ.setdefault('REDROB_GEMINI_FLASH_MODEL', 'gemini-3.5-flash')
    os.environ.setdefault('REDROB_GEMINI_PRO_MODEL', 'gemini-2.5-pro')
    print('Vertex AI project:', os.environ.get('GOOGLE_CLOUD_PROJECT', '(from ADC)'))
else:
    os.environ['REDROB_SKIP_GEMINI'] = '1'

In [ ]:
import sys
cmd = [sys.executable, 'colab/run_preprocess_gpu.py']
if FORCE:
    cmd.append('--force')
cmd.append('--no-skip-llm' if USE_GEMINI else '--skip-llm')
!{' '.join(cmd)}

In [ ]:
from google.colab import files
zip_path = PROJECT_ROOT / 'colab' / 'artifacts_download.zip'
assert zip_path.exists(), 'Run preprocess first — zip is created by run_preprocess_gpu.py'
files.download(str(zip_path))